In [ ]:
# for interactive plots
%matplotlib widget
# or
# %matplotlib ipympl
# for non-interactive plots
# %matplotlib inline

import warnings
warnings.filterwarnings('ignore')
import sunpy
import logging
sunpy.log.setLevel(logging.WARNING) # Set SunPy's logger to only show WARNING or above

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as colors
from matplotlib.ticker import AutoMinorLocator
from scipy.ndimage import gaussian_filter
from scipy.interpolate import UnivariateSpline
import astropy.io.fits as fits
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.visualization import ImageNormalize, LogStretch, LogStretch, PercentileInterval
from PIL import Image
import sunpy.map
from sunpy.time import parse_time
import sunpy.sun.constants as const
from sunkit_instruments import suvi
from scipy import stats
from copy import deepcopy
from tqdm import tqdm

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

data_dir = '/home/mnedal/data'

In [ ]:
date     = '2025-10-06'
# target   = '2025-10-06T08:58:05' # --> west
target   = '2025-10-06T08:45:00' # --> centre

CHANNELS = {'R': 211, 'G': 193, 'B': 171}   # colour -> AIA channel
year, month, day = date.split('-')
print(year, month, day)

ROIS = {
    'west_limb': dict(left=800,  right=1210, bottom=-300, top=300),
    'central':   dict(left=-120, right=500,  bottom=-200, top=400),
}

In [ ]:
def load_channel(channel):
    y, m, d = date.split('-')
    files = sorted(glob.glob(
        f'{data_dir}/AIA/{channel}A/highres/lv15/'
        f'aia.lev15.{channel}A_{y}_{m}_{d}T*_lev15*.fits'))
    if not files:
        raise FileNotFoundError(f'No files for {channel} A on {date}.')
    return sunpy.map.Map(files, sequence=True)


def running_ratio_map(sequence, target_time, lag=1, denom_floor=1.0):
    """Frame / previous frame at the nearest frame to target_time."""
    t   = parse_time(target_time)
    idx = int(np.argmin([abs(mp.date - t) for mp in sequence.maps]))
    if idx < lag:
        raise ValueError(f'Frame {idx} has no frame {lag} step(s) earlier.')
    cur, prev = sequence[idx], sequence[idx - lag]
    # NaN out near-zero/off-limb denominators so they map to black rather than blow up
    denom = np.where(prev.data > denom_floor, prev.data, np.nan)
    rmap  = sunpy.map.Map(cur.data / denom, cur.meta)
    rmap.plot_settings['norm'] = colors.Normalize()  # avoid inherited-stretch repr crash
    return rmap, cur.date


def crop_to_roi(rmap, box):
    frame = rmap.coordinate_frame
    bl = SkyCoord(box['left']  * u.arcsec, box['bottom'] * u.arcsec, frame=frame)
    tr = SkyCoord(box['right'] * u.arcsec, box['top']    * u.arcsec, frame=frame)
    return rmap.submap(bl, top_right=tr)


def scale_to_unit(data, method='percentile',
                  plo=0.5, phi=99.5, lo=0.5, hi=2.0, gamma=0.5):
    """Map a ratio array into [0, 1] for RGB display."""
    a = np.asarray(data, dtype=float)
    if method == 'percentile':
        vmin, vmax = np.nanpercentile(a, [plo, phi])
        out = (a - vmin) / (vmax - vmin)
    elif method == 'fixed':
        out = (a - lo) / (hi - lo)
    elif method == 'powernorm':
        out = colors.PowerNorm(gamma=gamma, vmin=lo, vmax=hi, clip=True)(a)
    else:
        raise ValueError("method must be 'percentile', 'fixed', or 'powernorm'.")
    return np.clip(np.nan_to_num(out, nan=0.0), 0, 1)


def build_rgb(r_map, g_map, b_map, method='percentile', **kw):
    # All three are AIA L1.5 on the same grid; trim to the common shape as a guard
    ny = min(m.data.shape[0] for m in (r_map, g_map, b_map))
    nx = min(m.data.shape[1] for m in (r_map, g_map, b_map))
    r = scale_to_unit(r_map.data[:ny, :nx], method, **kw)
    g = scale_to_unit(g_map.data[:ny, :nx], method, **kw)
    b = scale_to_unit(b_map.data[:ny, :nx], method, **kw)
    rgb = np.dstack([r, g, b])
    ref = sunpy.map.Map(g_map.data[:ny, :nx], g_map.meta)  # G carries the reference WCS
    ref.plot_settings['norm'] = colors.Normalize()
    return rgb, ref


def plot_rgb(rgb, ref_map, title=''):
    fig = plt.figure(figsize=[7,7])
    ax  = fig.add_subplot(projection=ref_map)
    ax.imshow(rgb, origin='lower')
    ref_map.draw_limb(axes=ax, color='white', alpha=0.5)
    ax.grid(False)
    ax.set_title(title)
    ax.set_xlabel('Solar X (arcsec)')
    ax.set_ylabel('Solar Y (arcsec)')
    return fig, ax

In [ ]:
# --- run ---
sequences = {c: load_channel(ch) for c, ch in CHANNELS.items()}

ratio_maps, times = {}, {}

with tqdm(total=len(CHANNELS), desc='Channels plotted') as pbar:
    for c in CHANNELS:
        ratio_maps[c], times[c] = running_ratio_map(sequences[c], target, lag=12)
        pbar.update(1)

print('Frame times:', {CHANNELS[c]: str(times[c]) for c in CHANNELS})

region = 'central'     # 'west_limb' or 'central'
method = 'powernorm'   # 'percentile' | 'fixed' | 'powernorm'

cropped = {c: crop_to_roi(ratio_maps[c], ROIS[region]) for c in CHANNELS}
rgb, ref = build_rgb(cropped['R'], cropped['G'], cropped['B'],
                     method=method, gamma=0.5, lo=0.5, hi=2)

In [ ]:
plot_rgb(rgb, ref,
         title=f'AIA 211/193/171 running ratio\n{region} — {times["G"]}')
plt.show()

In [ ]:
type(rgb)

Generic slit functions

In [ ]:
from scipy.ndimage import map_coordinates


def make_hpc_coord(x_arcsec, y_arcsec, map_obj):
    """
    Create a SkyCoord in the coordinate frame of map_obj.

    Parameters
    ----------
    x_arcsec, y_arcsec : float
        Helioprojective X/Y coordinates in arcsec.
    map_obj : sunpy.map.GenericMap
        Reference map whose coordinate frame is used.

    Returns
    -------
    SkyCoord
    """
    return SkyCoord(
        x_arcsec * u.arcsec,
        y_arcsec * u.arcsec,
        frame=map_obj.coordinate_frame
    )


def make_slit(start_xy, angle_deg, length_arcsec, map_obj):
    """
    Create one straight slit starting from an arbitrary HPC coordinate.

    Parameters
    ----------
    start_xy : tuple
        Starting point of slit as (x_arcsec, y_arcsec).
    angle_deg : float
        Angle measured counter-clockwise from solar West.
        0 deg   -> +X direction
        90 deg  -> +Y direction
        180 deg -> -X direction
    length_arcsec : float
        Slit length in arcsec.
    map_obj : sunpy.map.GenericMap
        Reference map.

    Returns
    -------
    SkyCoord
        Two-point SkyCoord containing start and end of slit.
    """
    x0, y0 = start_xy
    angle_rad = np.deg2rad(angle_deg)

    x1 = x0 + length_arcsec * np.cos(angle_rad)
    y1 = y0 + length_arcsec * np.sin(angle_rad)

    start = make_hpc_coord(x0, y0, map_obj)
    end   = make_hpc_coord(x1, y1, map_obj)

    return SkyCoord([start, end])


def make_slit_grid(start_xy, angles_deg, length_arcsec, map_obj):
    """
    Create multiple radial slits from the same starting point.

    Parameters
    ----------
    start_xy : tuple
        Starting coordinate as (x_arcsec, y_arcsec).
    angles_deg : array-like
        Slit angles in degrees.
    length_arcsec : float
        Slit length in arcsec.
    map_obj : sunpy.map.GenericMap

    Returns
    -------
    dict
        Dictionary of slit metadata.
    """
    slits = {}

    for i, angle in enumerate(angles_deg):
        slit_id = f'slit_{i:02d}'

        slits[slit_id] = {
            'angle_deg': angle,
            'start_xy': start_xy,
            'length_arcsec': length_arcsec,
            'line': make_slit(start_xy, angle, length_arcsec, map_obj)
        }

    return slits

Plot RGB with slits

In [ ]:
def plot_rgb_with_slits(rgb, ref_map, slits=None, title='',
                        slit_colour='red', slit_lw=1,
                        label_offset_arcsec=15):
    """
    Plot RGB image and optionally overlay slits.

    Slits are plotted as solid red lines.
    The slit number is shown near the slit tip instead of using a legend.
    """
    fig = plt.figure(figsize=[7,7])
    ax = fig.add_subplot(projection=ref_map)
    ax.imshow(rgb, origin='lower')
    ax.grid(False)

    if slits is not None:
        for slit_id, slit in slits.items():

            line = slit['line']

            ax.plot_coord(
                line,
                color=slit_colour,
                linestyle='-',
                lw=slit_lw,
                alpha=0.7
            )

            start = line[0]
            end = line[1]

            dx = end.Tx.to_value(u.arcsec) - start.Tx.to_value(u.arcsec)
            dy = end.Ty.to_value(u.arcsec) - start.Ty.to_value(u.arcsec)

            norm = np.sqrt(dx**2 + dy**2)

            if norm == 0:
                ux, uy = 0, 0
            else:
                ux, uy = dx / norm, dy / norm

            label_x = end.Tx.to_value(u.arcsec) + label_offset_arcsec * ux
            label_y = end.Ty.to_value(u.arcsec) + label_offset_arcsec * uy

            label_coord = SkyCoord(
                label_x * u.arcsec,
                label_y * u.arcsec,
                frame=ref_map.coordinate_frame
            )

            # Extract only the number from names like 'slit_03'
            slit_number = int(slit_id.split('_')[-1]) + 1
            
            ax.text_coord(
                label_coord,
                slit_number,
                color=slit_colour,
                fontsize=10,
                fontweight='bold',
                ha='center',
                va='center'
            )

    ax.set_title(title)
    ax.set_xlabel('Solar X [arcsec]')
    ax.set_ylabel('Solar Y [arcsec]')

    return fig, ax

Usage

In [ ]:
start_xy = (150, 100)      # user-defined slit origin in arcsec
length_arcsec = 280
angles_deg = np.arange(-90, 10, 10)

slits = make_slit_grid(
    start_xy=start_xy,
    angles_deg=angles_deg,
    length_arcsec=length_arcsec,
    map_obj=ref
)

plot_rgb_with_slits(
    rgb,
    ref,
    slits=slits,
    title=f'AIA 211/193/171 running ratio\n{region} — {times["G"]}'
)

plt.show()

Trace intensity along one slit

This samples a SunPy map along the slit using pixel coordinates and interpolation.

In [ ]:
def sample_map_along_slit(map_obj, slit_line, n_samples=300, order=1):
    """
    Extract intensity along a slit from a SunPy map.

    Parameters
    ----------
    map_obj : sunpy.map.GenericMap
        Map to sample.
    slit_line : SkyCoord
        Two-point SkyCoord defining the slit.
    n_samples : int
        Number of points along the slit.
    order : int
        Interpolation order for scipy.ndimage.map_coordinates.
        0 = nearest, 1 = linear.

    Returns
    -------
    distances_arcsec : ndarray
        Distance along slit in arcsec.
    intensities : ndarray
        Sampled intensity values.
    coords : SkyCoord
        Coordinates sampled along the slit.
    """
    start = slit_line[0]
    end   = slit_line[1]

    x_vals = np.linspace(start.Tx.to_value(u.arcsec),
                         end.Tx.to_value(u.arcsec),
                         n_samples)

    y_vals = np.linspace(start.Ty.to_value(u.arcsec),
                         end.Ty.to_value(u.arcsec),
                         n_samples)

    coords = SkyCoord(
        x_vals * u.arcsec,
        y_vals * u.arcsec,
        frame=map_obj.coordinate_frame
    )

    px, py = map_obj.world_to_pixel(coords)

    xpix = px.to_value(u.pix)
    ypix = py.to_value(u.pix)

    intensities = map_coordinates(
        map_obj.data.astype(float),
        [ypix, xpix],
        order=order,
        mode='constant',
        cval=np.nan
    )

    distances_arcsec = np.sqrt(
        (x_vals - x_vals[0])**2 +
        (y_vals - y_vals[0])**2
    )

    return distances_arcsec, intensities, coords

Process one RGB frame and trace all slits

This keeps your existing `running_ratio_map`, `crop_to_roi`, and `build_rgb`.

In [ ]:
def process_one_rgb_frame(sequences, target_time, channels, roi_box,
                          lag=12, method='powernorm', scale_kwargs=None):
    """
    Create one RGB running-ratio frame from 3 AIA channel sequences.

    Parameters
    ----------
    sequences : dict
        Example: {'R': seq_211, 'G': seq_193, 'B': seq_171}
    target_time : str
        Target time.
    channels : dict
        Example: {'R': 211, 'G': 193, 'B': 171}
    roi_box : dict
        ROI box with left/right/bottom/top in arcsec.
    lag : int
        Running-ratio lag.
    method : str
        Scaling method for RGB.
    scale_kwargs : dict or None
        Extra arguments for build_rgb.

    Returns
    -------
    result : dict
    """
    if scale_kwargs is None:
        scale_kwargs = {}

    ratio_maps = {}
    frame_times = {}

    for colour in channels:
        ratio_maps[colour], frame_times[colour] = running_ratio_map(
            sequences[colour],
            target_time,
            lag=lag
        )

    cropped = {
        colour: crop_to_roi(ratio_maps[colour], roi_box)
        for colour in channels
    }

    rgb, ref_map = build_rgb(
        cropped['R'],
        cropped['G'],
        cropped['B'],
        method=method,
        **scale_kwargs
    )

    return {
        'target_time': target_time,
        'frame_times': frame_times,
        'ratio_maps': ratio_maps,
        'cropped_maps': cropped,
        'rgb': rgb,
        'ref_map': ref_map
    }


def trace_slits_for_frame(frame_result, slits, n_samples=300):
    """
    Trace all slits for one processed RGB frame.

    Uses cropped running-ratio maps for R/G/B channels.

    Returns
    -------
    traces : dict
    """
    traces = {}

    for slit_id, slit in slits.items():
        traces[slit_id] = {
            'angle_deg': slit['angle_deg'],
            'start_xy': slit['start_xy'],
            'length_arcsec': slit['length_arcsec'],
            'channels': {}
        }

        for colour, mp in frame_result['cropped_maps'].items():
            distance, intensity, coords = sample_map_along_slit(
                mp,
                slit['line'],
                n_samples=n_samples
            )

            traces[slit_id]['channels'][colour] = {
                'distance_arcsec': distance,
                'intensity': intensity,
                'coords': coords
            }

    return traces

Process a full time sequence

In [ ]:
# You need a list of target times. Example:
target_times = pd.date_range(
    '2025-10-06T08:40:00',
    '2025-10-06T09:05:00',
    freq='24s'
).strftime('%Y-%m-%dT%H:%M:%S').tolist()

# Then process everything:
def process_rgb_slit_sequence(sequences, target_times, channels, roi_box,
                              start_xy, angles_deg, length_arcsec,
                              lag=12, method='powernorm',
                              scale_kwargs=None, n_samples=300):
    """
    Process many RGB frames and trace slits through all of them.

    Returns
    -------
    results : dict
    """
    if scale_kwargs is None:
        scale_kwargs = {}

    results = {
        'frames': {},
        'traces': {},
        'slits': None,
        'target_times': target_times
    }

    for i, target_time in enumerate(tqdm(target_times, desc='Processing RGB/slit sequence')):

        frame = process_one_rgb_frame(
            sequences=sequences,
            target_time=target_time,
            channels=channels,
            roi_box=roi_box,
            lag=lag,
            method=method,
            scale_kwargs=scale_kwargs
        )

        if results['slits'] is None:
            results['slits'] = make_slit_grid(
                start_xy=start_xy,
                angles_deg=angles_deg,
                length_arcsec=length_arcsec,
                map_obj=frame['ref_map']
            )

        traces = trace_slits_for_frame(
            frame_result=frame,
            slits=results['slits'],
            n_samples=n_samples
        )

        results['frames'][target_time] = frame
        results['traces'][target_time] = traces

    return results

Run it like this:

In [ ]:
sequences = {
    c: load_channel(ch)
    for c, ch in CHANNELS.items()
}

region = 'central'
roi_box = ROIS[region]

target_times = pd.date_range(
    '2025-10-06T08:40:00',
    '2025-10-06T09:05:00',
    freq='12s'
).strftime('%Y-%m-%dT%H:%M:%S').tolist()

start_xy = (150, 100)
angles_deg = np.arange(-90, 10, 10)
length_arcsec = 300

results = process_rgb_slit_sequence(
    sequences=sequences,
    target_times=target_times,
    channels=CHANNELS,
    roi_box=roi_box,
    start_xy=start_xy,
    angles_deg=angles_deg,
    length_arcsec=length_arcsec,
    lag=12,
    method='powernorm',
    scale_kwargs=dict(gamma=0.5, lo=0.5, hi=2),
    n_samples=300
)

Build a J-plot for one slit

This makes a distance-time image for one slit and one channel.

In [ ]:
def arcsec_to_Mm_factor(ref_map):
    """
    Convert arcsec to Mm using the observed solar radius from the map metadata.

    Uses:
        factor = physical solar radius / observed angular solar radius
    """
    rsun_obs_arcsec = ref_map.rsun_obs.to_value(u.arcsec)
    rsun_Mm = const.radius.to_value(u.Mm)

    return rsun_Mm / rsun_obs_arcsec

In [ ]:
def build_jplot(results, slit_id, channel='G', distance_unit='Mm'):
    """
    Build J-plot array for one slit and one colour channel.

    Parameters
    ----------
    results : dict
        Output from process_rgb_slit_sequence.

    slit_id : str
        Example: 'slit_00'.

    channel : str
        'R', 'G', or 'B'.

    distance_unit : {'arcsec', 'Mm'}
        Unit of returned slit distance.

    Returns
    -------
    jplot : ndarray
        Shape: [n_times, n_distances]

    times : list
        Frame times.

    distances : ndarray
        Distance along slit.

    distance_label : str
        Axis label.
    """
    times = list(results['traces'].keys())

    rows = []

    for t in times:
        trace = results['traces'][t][slit_id]['channels'][channel]
        rows.append(trace['intensity'])

    jplot = np.asarray(rows)

    distances_arcsec = results['traces'][times[0]][slit_id]['channels'][channel]['distance_arcsec']

    if distance_unit == 'arcsec':
        distances = distances_arcsec
        distance_label = 'Distance along slit [arcsec]'

    elif distance_unit == 'Mm':
        ref_map = results['frames'][times[0]]['ref_map']
        factor = arcsec_to_Mm_factor(ref_map)
        distances = distances_arcsec * factor
        distance_label = 'Distance along slit [Mm]'

    else:
        raise ValueError("distance_unit must be 'arcsec' or 'Mm'.")

    return jplot, times, distances, distance_label

Plot J-plot

In [ ]:
def plot_jplot(results, slit_id, channel='G',
               distance_unit='Mm',
               xlim=None, ylim=None,
               vmin=None, vmax=None, percentile=(1, 99),
               cmap='RdYlBu_r'):
    """
    Plot a J-plot for one slit.

    X-axis: time
    Y-axis: distance along slit, either arcsec or Mm.
    """
    jplot, times, distances, distance_label = build_jplot(
        results,
        slit_id=slit_id,
        channel=channel,
        distance_unit=distance_unit
    )

    time_nums = mdates.date2num(pd.to_datetime(times))

    if vmin is None or vmax is None:
        vmin, vmax = np.nanpercentile(jplot, percentile)

    fig, ax = plt.subplots(figsize=[8,6])

    mesh = ax.pcolormesh(
        time_nums,
        distances,
        jplot.T,
        shading='auto',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax
    )

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    if xlim is not None:
        left, right = xlim
        left_num = mdates.date2num(pd.to_datetime(left))
        right_num = mdates.date2num(pd.to_datetime(right))
        ax.set_xlim(left_num, right_num)

    if ylim is not None:
        bottom, top = ylim
        ax.set_ylim(bottom, top)

    ax.set_xlabel('Time [UT]')
    ax.set_ylabel(distance_label)
    ax.set_title(f'{slit_id} | channel {channel}')

    fig.colorbar(mesh, ax=ax, pad=0.01, label='Running-ratio intensity')

    fig.autofmt_xdate()

    return fig, ax, jplot, times, distances

Usage:

In [ ]:
jplot_left   = '2025-10-06T08:40:00'
jplot_right  = '2025-10-06T08:55:00'

jplot_bottom = 50
jplot_top    = 200


fig, ax, jplot, jtimes, distances, = plot_jplot(
    results,
    slit_id='slit_03',
    channel='G',
    xlim=(jplot_left, jplot_right),
    ylim=(jplot_bottom, jplot_top),
    cmap='Greys_r'
)

plt.show()

Collect repeated traces on the J-plot

In [ ]:
def collect_repeated_jplot_traces(fig, ax, slit_id, feature_id='front_00',
                                  n_repeats=5, storage=None, y_unit='Mm',
                                  marker='x', line_style='-'):
    """
    Interactively collect repeated traces of a moving feature on a J-plot.

    Mouse controls
    --------------
    Left click:
        Add a point to the current trace.

    Right click:
        Finish the current trace, store it, clear it from the plot,
        and start the next repetition.

    After n_repeats right-clicks, the function disconnects automatically.

    Parameters
    ----------
    fig, ax : matplotlib Figure and Axes
        J-plot figure and axis.

    slit_id : str
        Slit identifier, e.g. 'slit_03'.

    feature_id : str
        Name of the traced feature.

    n_repeats : int
        Number of repeated traces.

    storage : dict or None
        Existing storage dictionary. If None, a new one is created.

    marker : str
        Marker used for clicked points.

    line_style : str
        Line style used to connect clicked points.

    Returns
    -------
    storage : dict
        Dictionary containing all repeated traces.
    """
    if storage is None:
        storage = {}

    if slit_id not in storage:
        storage[slit_id] = {}

    if feature_id not in storage[slit_id]:
        storage[slit_id][feature_id] = {
            'y_unit': y_unit,
            'repeats': []
        }

    current_trace = {
        'x_time_num': [],
        'x_time': [],
        'y_distance': []
    }

    current_artists = []

    def redraw_current_trace():
        nonlocal current_artists

        for artist in current_artists:
            artist.remove()

        current_artists = []

        x = current_trace['x_time_num']
        y = current_trace['y_distance']

        if len(x) == 0:
            fig.canvas.draw_idle()
            return

        point_artist, = ax.plot(
            x,
            y,
            linestyle='None',
            marker=marker,
            ms=7,
            mew=1.5,
            color='red'
        )

        current_artists.append(point_artist)

        if len(x) > 1:
            line_artist, = ax.plot(
                x,
                y,
                linestyle=line_style,
                lw=1.5,
                alpha=0.8,
                color='red'
            )
            current_artists.append(line_artist)

        fig.canvas.draw_idle()

    def clear_current_trace_from_plot():
        nonlocal current_artists

        for artist in current_artists:
            artist.remove()

        current_artists = []
        fig.canvas.draw_idle()

    def reset_current_trace():
        current_trace['x_time_num'] = []
        current_trace['x_time'] = []
        current_trace['y_distance'] = []

    def onclick(event):
        nonlocal cid

        if event.inaxes != ax:
            return

        if event.xdata is None or event.ydata is None:
            return

        repeats = storage[slit_id][feature_id]['repeats']
        repeat_number = len(repeats) + 1

        if event.button == 1:
            x_num = event.xdata
            y_val = event.ydata
            x_time = mdates.num2date(x_num)

            current_trace['x_time_num'].append(x_num)
            current_trace['x_time'].append(x_time)
            current_trace['y_distance'].append(y_val)

            redraw_current_trace()

            print(
                f'{feature_id} | {slit_id} | '
                f'repeat {repeat_number}/{n_repeats} | '
                f'point {len(current_trace["x_time_num"])}: '
                f'time={x_time}, distance={y_val:.2f} {y_unit}'
            )

        elif event.button == 3:
            if len(current_trace['x_time_num']) == 0:
                print('No points in current trace. Left-click points first.')
                return

            repeats.append({
                'x_time_num': current_trace['x_time_num'].copy(),
                'x_time': current_trace['x_time'].copy(),
                'y_distance': current_trace['y_distance'].copy()
            })

            print(
                f'Saved repeat {len(repeats)}/{n_repeats} '
                f'for {feature_id} on {slit_id} '
                f'with {len(current_trace["x_time_num"])} points.'
            )

            clear_current_trace_from_plot()
            reset_current_trace()

            if len(repeats) >= n_repeats:
                fig.canvas.mpl_disconnect(cid)
                print(
                    f'Finished collecting {n_repeats} repeated traces '
                    f'for {feature_id} on {slit_id}.'
                )
            else:
                print(
                    f'Ready for repeat {len(repeats) + 1}/{n_repeats}. '
                    f'Left-click the feature again, then right-click to save.'
                )

    cid = fig.canvas.mpl_connect('button_press_event', onclick)

    print(
        f'Collecting {n_repeats} repeated traces for {feature_id} on {slit_id}.\n'
        f'Left click = add point.\n'
        f'Right click = save this trace and reset the J-plot.'
    )

    return storage

Usage:

In [ ]:
clicked_traces = {}

fig, ax, jplot, jtimes, distances = plot_jplot(
    results,
    slit_id='slit_03',
    channel='G',
    distance_unit='Mm',
    xlim=(jplot_left, jplot_right),
    ylim=(jplot_bottom, jplot_top),
    cmap='Greys_r'
)

clicked_traces = collect_repeated_jplot_traces(
    fig,
    ax,
    slit_id='slit_03',
    feature_id='front_00',
    n_repeats=5,
    storage=clicked_traces
)

plt.show()

In [ ]:
len(clicked_traces['slit_03']['front_00']['repeats'])

In [ ]:
clicked_traces

Estimate mean trace and standard error

In [ ]:
def summarise_repeated_traces(clicked_traces, slit_id, feature_id,
                              jplot_times,
                              min_repeats=2):
    """
    Compute mean and standard error from repeated manually clicked traces.

    The repeated traces are interpolated onto the actual J-plot time grid,
    not an arbitrary synthetic grid.

    Parameters
    ----------
    clicked_traces : dict
        Output from collect_repeated_jplot_traces.

    slit_id : str
        Slit identifier.

    feature_id : str
        Feature identifier.

    jplot_times : list or array-like
        Times returned by plot_jplot/build_jplot.

    min_repeats : int
        Minimum number of valid repeated traces required.

    Returns
    -------
    summary : dict
    """
    repeats = clicked_traces[slit_id][feature_id]['repeats']
    y_unit = clicked_traces[slit_id][feature_id].get('y_unit', '')

    valid_repeats = []

    for rep in repeats:
        x = np.asarray(rep['x_time_num'], dtype=float)
        y = np.asarray(rep['y_distance'], dtype=float)

        good = np.isfinite(x) & np.isfinite(y)
        x = x[good]
        y = y[good]

        if len(x) < 2:
            continue

        order = np.argsort(x)
        x = x[order]
        y = y[order]

        x_unique, unique_idx = np.unique(x, return_index=True)
        y_unique = y[unique_idx]

        if len(x_unique) < 2:
            continue

        valid_repeats.append({
            'x_time_num': x_unique,
            'y_distance': y_unique
        })

    if len(valid_repeats) < min_repeats:
        raise ValueError(
            f'Need at least {min_repeats} valid repeated traces. '
            f'Only found {len(valid_repeats)}.'
        )

    jplot_time_nums = mdates.date2num(pd.to_datetime(jplot_times))

    x_left = max(rep['x_time_num'][0] for rep in valid_repeats)
    x_right = min(rep['x_time_num'][-1] for rep in valid_repeats)

    x_grid = jplot_time_nums[
        (jplot_time_nums >= x_left) &
        (jplot_time_nums <= x_right)
    ]

    if len(x_grid) < 2:
        raise ValueError(
            'The repeated traces overlap over fewer than two J-plot frames. '
            'Trace over a wider or more consistent time interval.'
        )

    y_interp = []

    for rep in valid_repeats:
        y_grid = np.interp(
            x_grid,
            rep['x_time_num'],
            rep['y_distance']
        )
        y_interp.append(y_grid)

    y_interp = np.asarray(y_interp)

    y_mean = np.nanmean(y_interp, axis=0)
    y_std = np.nanstd(y_interp, axis=0, ddof=1)
    y_sem = y_std / np.sqrt(y_interp.shape[0])

    summary = {
        'slit_id': slit_id,
        'feature_id': feature_id,
        'y_unit': y_unit,
        'n_repeats': len(valid_repeats),
        'x_time_num': x_grid,
        'x_time': [mdates.num2date(x) for x in x_grid],
        'y_distance_mean': y_mean,
        'y_distance_std': y_std,
        'y_distance_sem': y_sem,
        'y_repeats_interp': y_interp
    }

    return summary

Usage:

In [ ]:
trace_summary = summarise_repeated_traces(
    clicked_traces,
    slit_id='slit_03',
    feature_id='front_00',
    jplot_times=jtimes
)

Plot the mean trace with standard error

In [ ]:
def overplot_trace_summary(ax, trace_summary,
                           errorbar_every=1,
                           mean_lw=2.0,
                           marker='o',
                           capsize=3):
    """
    Overplot mean traced feature with SEM error bars on the J-plot.
    """
    x = np.asarray(trace_summary['x_time_num'])
    y = np.asarray(trace_summary['y_distance_mean'])
    yerr = np.asarray(trace_summary['y_distance_sem'])

    idx = np.arange(len(x))[::errorbar_every]

    ax.errorbar(
        x[idx],
        y[idx],
        yerr=yerr[idx],
        fmt=marker + '-',
        lw=mean_lw,
        ms=4,
        capsize=capsize,
        color='red',
        label=f'{trace_summary["feature_id"]} mean ± SEM'
    )

    ax.legend()

    return ax


# def overplot_trace_summary(ax, trace_summary,
#                            show_sem=True,
#                            mean_lw=2.5,
#                            sem_alpha=0.25):
#     """
#     Overplot mean traced feature and standard error on the J-plot.
#     """
#     x = trace_summary['x_time_num']
#     y = trace_summary['y_distance_arcsec_mean']
#     y_sem = trace_summary['y_distance_arcsec_sem']

#     ax.plot(
#         x,
#         y,
#         lw=mean_lw,
#         label=f'{trace_summary["feature_id"]} mean'
#     )

#     if show_sem:
#         ax.fill_between(
#             x,
#             y - y_sem,
#             y + y_sem,
#             alpha=sem_alpha,
#             label='SEM' # standard error of the mean
#         )

#     ax.legend()

#     return ax

Usage:

In [ ]:
fig, ax, jplot, jtimes, distances_Mm = plot_jplot(
    results,
    slit_id='slit_03',
    channel='G',
    distance_unit='Mm',
    xlim=(jplot_left, jplot_right),
    ylim=(jplot_bottom, jplot_top),
    cmap='Greys_r'
)

overplot_trace_summary(
    ax,
    trace_summary,
    errorbar_every=2
)

plt.show()


# fig, ax, jplot, jtimes, distances = plot_jplot(
#     results,
#     slit_id='slit_03',
#     channel='G',
#     xlim=(jplot_left, jplot_right),
#     ylim=(jplot_bottom, jplot_top),
#     cmap='Greys_r'
# )

# overplot_trace_summary(ax, trace_summary)
# plt.show()

Convert the summary to a DataFrame

In [ ]:
def trace_summary_to_dataframe(trace_summary):
    """
    Convert one repeated-trace summary to a DataFrame.
    """
    df = pd.DataFrame({
        'slit_id': trace_summary['slit_id'],
        'feature_id': trace_summary['feature_id'],
        'n_repeats': trace_summary['n_repeats'],
        'time': trace_summary['x_time'],
        'time_num': trace_summary['x_time_num'],
        f'distance_mean_{trace_summary["y_unit"]}': trace_summary['y_distance_mean'],
        f'distance_std_{trace_summary["y_unit"]}': trace_summary['y_distance_std'],
        f'distance_sem_{trace_summary["y_unit"]}': trace_summary['y_distance_sem']
    })

    return df

Usage:

In [ ]:
df_trace = trace_summary_to_dataframe(trace_summary)

df_trace.to_csv(
    './slit_03_front_00_repeated_trace_summary.csv',
    index=False
)

df_trace.head()

More generic; applicable for a choosen EUV channel

In [ ]:
for channel in ['R', 'G', 'B']:
    fig, ax, jplot, jtimes, distances_Mm = plot_jplot(
        results,
        slit_id='slit_03',
        channel=channel,
        distance_unit='Mm',
        xlim=(jplot_left, jplot_right),
        ylim=(jplot_bottom, jplot_top),
        cmap='Greys_r'
    )

    plt.show()

In [ ]:
clicked_traces = {}

slit_id = 'slit_03'
feature_id = 'front_00'
channel = 'G'

fig, ax, jplot, jtimes, distances_Mm = plot_jplot(
    results,
    slit_id=slit_id,
    channel=channel,
    distance_unit='Mm',
    xlim=(jplot_left, jplot_right),
    ylim=(jplot_bottom, jplot_top),
    cmap='Greys_r'
)

clicked_traces = collect_repeated_jplot_traces(
    fig,
    ax,
    slit_id=slit_id,
    feature_id=feature_id,
    n_repeats=5,
    storage=clicked_traces,
    y_unit='Mm'
)

plt.show()

In [ ]:
trace_summary = summarise_repeated_traces(
    clicked_traces,
    slit_id=slit_id,
    feature_id=feature_id,
    jplot_times=jtimes
)

fig, ax, jplot, jtimes, distances_Mm = plot_jplot(
    results,
    slit_id=slit_id,
    channel=channel,
    distance_unit='Mm',
    xlim=(jplot_left, jplot_right),
    ylim=(jplot_bottom, jplot_top),
    cmap='Greys_r'
)

overplot_trace_summary(
    ax,
    trace_summary,
    errorbar_every=2
)

plt.show()

df_trace = trace_summary_to_dataframe(trace_summary)

df_trace.to_csv(
    f'./{slit_id}_{feature_id}_{channel}_trace_summary.csv',
    index=False
)

Estimate the kinematics

In [ ]:
from scipy.signal import savgol_filter


def get_valid_savgol_window(n_points, requested_window=7, polyorder=2):
    """
    Return a valid odd Savitzky-Golay window length.

    The window must be:
    - odd
    - <= n_points
    - > polyorder
    """
    if n_points <= polyorder + 1:
        raise ValueError(
            f'Need more than {polyorder + 1} points for Savitzky-Golay smoothing.'
        )

    window = min(requested_window, n_points)

    if window % 2 == 0:
        window -= 1

    if window <= polyorder:
        window = polyorder + 2

        if window % 2 == 0:
            window += 1

    if window > n_points:
        window = n_points

        if window % 2 == 0:
            window -= 1

    return window

In [ ]:
def compute_trace_kinematics(trace_summary,
                             smooth=True,
                             savgol_window=7,
                             savgol_polyorder=2):
    """
    Compute distance, speed, and acceleration from repeated traced points.

    Parameters
    ----------
    trace_summary : dict
        Output from summarise_repeated_traces().

        Required keys:
            'x_time_num'
            'x_time'
            'y_repeats_interp'
            'y_unit'

    smooth : bool
        If True, smooth each repeated distance-time trace before
        differentiating.

    savgol_window : int
        Requested Savitzky-Golay window length.

    savgol_polyorder : int
        Polynomial order for Savitzky-Golay filter.

    Returns
    -------
    kin : dict
        Kinematics dictionary containing distance, speed, and acceleration
        with uncertainties.
    """
    x_time_num = np.asarray(trace_summary['x_time_num'], dtype=float)
    x_time = trace_summary['x_time']

    y_repeats = np.asarray(trace_summary['y_repeats_interp'], dtype=float)

    if y_repeats.ndim != 2:
        raise ValueError('trace_summary["y_repeats_interp"] must be a 2D array.')

    n_repeats, n_times = y_repeats.shape

    if n_repeats < 2:
        raise ValueError('Need at least two repeated traces to estimate uncertainty.')

    if n_times < 3:
        raise ValueError('Need at least three time points to estimate speed and acceleration.')

    # Matplotlib date numbers are in days.
    # Convert to seconds relative to first time.
    t_seconds = (x_time_num - x_time_num[0]) * 24 * 3600

    # Distance unit conversion.
    # If y is in Mm:
    # speed = Mm/s * 1000 -> km/s
    # acceleration = Mm/s^2 * 1e6 -> m/s^2
    y_unit = trace_summary.get('y_unit', 'Mm')

    if y_unit == 'Mm':
        speed_factor = 1000.0
        acceleration_factor = 1e6
        distance_unit = 'Mm'
        speed_unit = 'km/s'
        acceleration_unit = 'm/s^2'

    elif y_unit == 'arcsec':
        raise ValueError(
            'Kinematics should be computed in physical distance units. '
            'Use distance_unit="Mm" when creating the J-plot/traces.'
        )

    else:
        raise ValueError(f'Unsupported y_unit: {y_unit}')

    distance_repeats = []
    speed_repeats = []
    acceleration_repeats = []

    if smooth:
        window = get_valid_savgol_window(
            n_points=n_times,
            requested_window=savgol_window,
            polyorder=savgol_polyorder
        )

    for i in range(n_repeats):

        y = y_repeats[i]

        if smooth:
            y_use = savgol_filter(
                y,
                window_length=window,
                polyorder=savgol_polyorder,
                mode='interp'
            )
        else:
            y_use = y.copy()

        # First derivative: Mm/s
        v = np.gradient(y_use, t_seconds)

        # Second derivative: Mm/s^2
        a = np.gradient(v, t_seconds)

        # Convert units
        v = v * speed_factor
        a = a * acceleration_factor

        distance_repeats.append(y_use)
        speed_repeats.append(v)
        acceleration_repeats.append(a)

    distance_repeats = np.asarray(distance_repeats)
    speed_repeats = np.asarray(speed_repeats)
    acceleration_repeats = np.asarray(acceleration_repeats)

    distance_mean = np.nanmean(distance_repeats, axis=0)
    distance_std = np.nanstd(distance_repeats, axis=0, ddof=1)
    distance_sem = distance_std / np.sqrt(n_repeats)

    speed_mean = np.nanmean(speed_repeats, axis=0)
    speed_std = np.nanstd(speed_repeats, axis=0, ddof=1)
    speed_sem = speed_std / np.sqrt(n_repeats)

    acceleration_mean = np.nanmean(acceleration_repeats, axis=0)
    acceleration_std = np.nanstd(acceleration_repeats, axis=0, ddof=1)
    acceleration_sem = acceleration_std / np.sqrt(n_repeats)

    kin = {
        'slit_id': trace_summary['slit_id'],
        'feature_id': trace_summary['feature_id'],
        'n_repeats': n_repeats,

        'x_time_num': x_time_num,
        'x_time': x_time,
        't_seconds': t_seconds,

        'distance_unit': distance_unit,
        'speed_unit': speed_unit,
        'acceleration_unit': acceleration_unit,

        'distance_repeats': distance_repeats,
        'speed_repeats': speed_repeats,
        'acceleration_repeats': acceleration_repeats,

        'distance_mean': distance_mean,
        'distance_std': distance_std,
        'distance_sem': distance_sem,

        'speed_mean': speed_mean,
        'speed_std': speed_std,
        'speed_sem': speed_sem,

        'acceleration_mean': acceleration_mean,
        'acceleration_std': acceleration_std,
        'acceleration_sem': acceleration_sem,
    }

    return kin

In [ ]:
def plot_trace_kinematics(kin,
                          errorbar_every=1,
                          show_distance=True,
                          show_speed=True,
                          show_acceleration=True):
    """
    Plot distance, speed, and acceleration versus time with SEM error bars.

    Parameters
    ----------
    kin : dict
        Output from compute_trace_kinematics().

    errorbar_every : int
        Plot one error bar every N points to avoid visual clutter.
    """
    x = np.asarray(kin['x_time_num'])
    idx = np.arange(len(x))[::errorbar_every]

    figs_axes = {}

    if show_distance:
        fig, ax = plt.subplots(figsize=[8, 4])

        ax.errorbar(
            x[idx],
            kin['distance_mean'][idx],
            yerr=kin['distance_sem'][idx],
            fmt='o-',
            ms=4,
            lw=1.5,
            capsize=3,
            label='Distance mean ± SEM'
        )

        ax.xaxis_date()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        ax.set_xlabel('Time [UT]')
        ax.set_ylabel(f'Distance [{kin["distance_unit"]}]')
        ax.set_title(f'{kin["slit_id"]} | {kin["feature_id"]} | distance')
        ax.legend()
        fig.autofmt_xdate()

        figs_axes['distance'] = (fig, ax)

    if show_speed:
        fig, ax = plt.subplots(figsize=[8, 4])

        ax.errorbar(
            x[idx],
            kin['speed_mean'][idx],
            yerr=kin['speed_sem'][idx],
            fmt='o-',
            ms=4,
            lw=1.5,
            capsize=3,
            label='Speed mean ± SEM'
        )

        ax.axhline(0, color='black', lw=0.8, alpha=0.5)

        ax.xaxis_date()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        ax.set_xlabel('Time [UT]')
        ax.set_ylabel(f'Speed [{kin["speed_unit"]}]')
        ax.set_title(f'{kin["slit_id"]} | {kin["feature_id"]} | speed')
        ax.legend()
        fig.autofmt_xdate()

        figs_axes['speed'] = (fig, ax)

    if show_acceleration:
        fig, ax = plt.subplots(figsize=[8, 4])

        ax.errorbar(
            x[idx],
            kin['acceleration_mean'][idx],
            yerr=kin['acceleration_sem'][idx],
            fmt='o-',
            ms=4,
            lw=1.5,
            capsize=3,
            label='Acceleration mean ± SEM'
        )

        ax.axhline(0, color='black', lw=0.8, alpha=0.5)

        ax.xaxis_date()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

        ax.set_xlabel('Time [UT]')
        ax.set_ylabel(f'Acceleration [{kin["acceleration_unit"]}]')
        ax.set_title(f'{kin["slit_id"]} | {kin["feature_id"]} | acceleration')
        ax.legend()
        fig.autofmt_xdate()

        figs_axes['acceleration'] = (fig, ax)

    return figs_axes

In [ ]:
def kinematics_to_dataframe(kin):
    """
    Convert kinematics dictionary to a pandas DataFrame.
    """
    df = pd.DataFrame({
        'slit_id': kin['slit_id'],
        'feature_id': kin['feature_id'],
        'n_repeats': kin['n_repeats'],

        'time': kin['x_time'],
        'time_num': kin['x_time_num'],
        't_seconds': kin['t_seconds'],

        f'distance_mean_{kin["distance_unit"]}': kin['distance_mean'],
        f'distance_std_{kin["distance_unit"]}': kin['distance_std'],
        f'distance_sem_{kin["distance_unit"]}': kin['distance_sem'],

        f'speed_mean_{kin["speed_unit"]}': kin['speed_mean'],
        f'speed_std_{kin["speed_unit"]}': kin['speed_std'],
        f'speed_sem_{kin["speed_unit"]}': kin['speed_sem'],

        f'acceleration_mean_{kin["acceleration_unit"]}': kin['acceleration_mean'],
        f'acceleration_std_{kin["acceleration_unit"]}': kin['acceleration_std'],
        f'acceleration_sem_{kin["acceleration_unit"]}': kin['acceleration_sem'],
    })

    return df

Full usage for one slit

In [ ]:
trace_summary = summarise_repeated_traces(
    clicked_traces,
    slit_id='slit_03',
    feature_id='front_00',
    jplot_times=jtimes
)

In [ ]:
kin = compute_trace_kinematics(
    trace_summary,
    smooth=True,
    savgol_window=7,
    savgol_polyorder=2
)

figs_axes = plot_trace_kinematics(
    kin,
    errorbar_every=1
)

plt.show()

In [ ]:
df_kin = kinematics_to_dataframe(kin)

df_kin.to_csv(
    './slit_03_front_00_kinematics.csv',
    index=False
)

df_kin.head()

Process many slits

Assuming you have traced the same feature for several slits:

In [ ]:
slit_ids = ['slit_00', 'slit_01', 'slit_02', 'slit_03']
feature_id = 'front_00'

In [ ]:
def compute_kinematics_for_slits(clicked_traces, slit_ids, feature_id,
                                 jplot_times_by_slit,
                                 smooth=True,
                                 savgol_window=7,
                                 savgol_polyorder=2):
    """
    Compute kinematics for several slits.

    Parameters
    ----------
    clicked_traces : dict
        Manual traced points.

    slit_ids : list
        List of slit IDs.

    feature_id : str
        Feature name.

    jplot_times_by_slit : dict
        Dictionary giving J-plot times for each slit.

        Example:
        jplot_times_by_slit['slit_03'] = jtimes

    Returns
    -------
    all_kin : dict
        Kinematics per slit.
    """
    all_kin = {}

    for slit_id in slit_ids:

        trace_summary = summarise_repeated_traces(
            clicked_traces,
            slit_id=slit_id,
            feature_id=feature_id,
            jplot_times=jplot_times_by_slit[slit_id]
        )

        kin = compute_trace_kinematics(
            trace_summary,
            smooth=smooth,
            savgol_window=savgol_window,
            savgol_polyorder=savgol_polyorder
        )

        all_kin[slit_id] = kin

    return all_kin

Usage:

In [ ]:
jplot_times_by_slit = {}

for slit_id in slit_ids:
    fig, ax, jplot, jtimes, distances_Mm = plot_jplot(
        results,
        slit_id=slit_id,
        channel='G',
        distance_unit='Mm',
        xlim=(jplot_left, jplot_right),
        ylim=(jplot_bottom, jplot_top),
        cmap='Greys_r'
    )

    jplot_times_by_slit[slit_id] = jtimes

In [ ]:
all_kin = compute_kinematics_for_slits(
    clicked_traces=clicked_traces,
    slit_ids=slit_ids,
    feature_id='front_00',
    jplot_times_by_slit=jplot_times_by_slit,
    smooth=True,
    savgol_window=7,
    savgol_polyorder=2
)

Save all kinematics into one table:

In [ ]:
df_all_kin = pd.concat(
    [kinematics_to_dataframe(all_kin[slit_id]) for slit_id in all_kin],
    ignore_index=True
)

df_all_kin.to_csv(
    './all_slits_front_00_kinematics.csv',
    index=False
)

df_all_kin.head()

Plot speed/acceleration for all slits together

In [ ]:
def plot_all_slit_speeds(all_kin, errorbar_every=1):
    """
    Plot speed versus time for all slits.
    """
    fig, ax = plt.subplots(figsize=[9, 5])

    for slit_id, kin in all_kin.items():
        x = np.asarray(kin['x_time_num'])
        idx = np.arange(len(x))[::errorbar_every]

        ax.errorbar(
            x[idx],
            kin['speed_mean'][idx],
            yerr=kin['speed_sem'][idx],
            fmt='o-',
            ms=3,
            lw=1.2,
            capsize=2,
            label=slit_id
        )

    ax.axhline(0, color='black', lw=0.8, alpha=0.5)

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    ax.set_xlabel('Time [UT]')
    ax.set_ylabel('Speed [km/s]')
    ax.set_title('Speed along slits')
    ax.legend(fontsize=8)

    fig.autofmt_xdate()

    return fig, ax


def plot_all_slit_accelerations(all_kin, errorbar_every=1):
    """
    Plot acceleration versus time for all slits.
    """
    fig, ax = plt.subplots(figsize=[9, 5])

    for slit_id, kin in all_kin.items():
        x = np.asarray(kin['x_time_num'])
        idx = np.arange(len(x))[::errorbar_every]

        ax.errorbar(
            x[idx],
            kin['acceleration_mean'][idx],
            yerr=kin['acceleration_sem'][idx],
            fmt='o-',
            ms=3,
            lw=1.2,
            capsize=2,
            label=slit_id
        )

    ax.axhline(0, color='black', lw=0.8, alpha=0.5)

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    ax.set_xlabel('Time [UT]')
    ax.set_ylabel('Acceleration [m/s$^2$]')
    ax.set_title('Acceleration along slits')
    ax.legend(fontsize=8)

    fig.autofmt_xdate()

    return fig, ax

Usage:

In [ ]:
fig, ax = plot_all_slit_speeds(
    all_kin,
    errorbar_every=1
)

plt.show()

In [ ]:
fig, ax = plot_all_slit_accelerations(
    all_kin,
    errorbar_every=1
)

plt.show()

Recommended final workflow

In [ ]:
slit_id = 'slit_03'
feature_id = 'front_00'

trace_summary = summarise_repeated_traces(
    clicked_traces,
    slit_id=slit_id,
    feature_id=feature_id,
    jplot_times=jtimes
)

kin = compute_trace_kinematics(
    trace_summary,
    smooth=True,
    savgol_window=7,
    savgol_polyorder=2
)

plot_trace_kinematics(
    kin,
    errorbar_every=1
)

df_kin = kinematics_to_dataframe(kin)

df_kin.to_csv(
    f'./{slit_id}_{feature_id}_kinematics.csv',
    index=False
)